# 02 - Character Segmentation (crops Viola-Jones)

Aquest notebook processa els **crops reals** generats pel detector Viola-Jones (VJ) i en segmenta els caràcters individuals. Respecte a la versió original, incorpora quatre millores:

1. **`RETR_EXTERNAL` en lloc de `RETR_TREE`** — evita que els forats interiors de caràcters com 'O', '0', 'D', 'A', 'B' es detectin com a contorns separats i es comptin dues vegades.
2. **Morfologia CLOSE vertical** (`1×3` px) — consolida traços verticals fins (especialment la 'I') sense fusionar caràcters adjacents.
3. **Deduplicació IoU** — xarxa de seguretat per eliminar bboxes que solapen excessivament (IoU > 0.5) tot conservant el contorn exterior (el de més àrea).
4. **Validació geomètrica** (`is_plausible_plate`) — rebutja crops on els caràcters detectats no compleixen la coherència geomètrica d'una matrícula real (uniformitat d'alçades, alineació vertical, cobertura horitzontal).

**Flux de processament per crop:**
```
crop_bgr
  → deskew (minAreaRect)
  → adaptiveThreshold + MORPH_CLOSE (1×3)
  → findContours (RETR_EXTERNAL)
  → filtre alçada mediana + AR [0.05, 0.95]
  → deduplicació IoU
  → is_plausible_plate (count + uniformitat + alineació + cobertura)
  → resize 28×28 → guardar
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
import os


def mostrar_imatge(titol, imatge, cmap=None):
    plt.figure(figsize=(10, 4))
    plt.title(titol)
    if cmap:
        plt.imshow(imatge, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(imatge, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


def _glob_crops(directory):
    """Retorna tots els crops disponibles (suporta _cand*.jpg i _box*.png)."""
    return sorted(
        list(directory.glob('*_cand*.jpg')) +
        list(directory.glob('*_box*.png'))
    )


# ── Directoris ────────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('data/processed')   # crops del VJ: {stem}_cand{n}.jpg
OUTPUT_DIR    = Path('data/chars')       # sortida: {stem}_cand{n}_char{i}.png

# ── Rang de caràcters vàlids per a una matrícula ──────────────────────────────
N_CHARS_MIN = 5
N_CHARS_MAX = 8

# ── Mida d'entrada per a la CNN ───────────────────────────────────────────────
CNN_INPUT_W = 28
CNN_INPUT_H = 28

# ── Límit de seguretat per al deskew ─────────────────────────────────────────
ALIGN_ANGLE_MAX = 15.0   # graus: si |angle| supera aquest valor no rotem

# ── Kernel morfològic CLOSE (problema 3: traços verticals fins / 'I') ─────────
# Vertical 1×3: consolida traços dins d'un caràcter sense unir caràcters veïns.
# Un kernel quadrat gran fusionaria caràcters adjacents.
MORPH_KERNEL_SIZE = (1, 3)

# ── Deduplicació IoU (problema 1: forats interiors que passin el filtre) ──────
# Si dos bboxes solapen més d'aquest valor, eliminem el de menys àrea (el forat).
IOU_OVERLAP_THRESH = 0.50

# ── Llindars per a is_plausible_plate (problema 2: falsos positius) ───────────
# Desviació estàndard màxima d'alçades com a fracció de la mediana.
HEIGHT_STD_MAX     = 0.20
# Desviació màxima dels centroides verticals (cy) com a fracció de la mediana.
ALIGN_CY_STD_MAX   = 0.25
# Fracció mínima de l'amplada del crop coberta pels caràcters.
MIN_WIDTH_COVERAGE = 0.30

EXAMPLE_INDEX = 0   # índex de crop per a les cel·les de demostració
N_DEBUG       = 20  # crops mostrats a la secció de debug

print("Llibreries carregades.")
print(f"Directori d'entrada : {PROCESSED_DIR}")
print(f"Directori de sortida: {OUTPUT_DIR}")
print(f"Rang de caràcters   : [{N_CHARS_MIN}, {N_CHARS_MAX}]")
print(f"Crops disponibles   : {len(_glob_crops(PROCESSED_DIR))}")

## Pas 0: Correcció d'inclinació (deskew)

Els crops del detector VJ no estan garantidament horitzontals. Alineiem el crop:

1. Convertim a grisos i binaritzem amb Otsu.
2. Assegurem que els caràcters siguin **blancs**.
3. `cv2.minAreaRect` sobre tots els píxels blancs → angle d'inclinació.
4. Convertim l'angle: si `w < h` → `angle + 90°`; si `w ≥ h` → `angle` directament.
5. Si `|angle| > ALIGN_ANGLE_MAX` no rotem.
6. `cv2.warpAffine` amb `BORDER_REPLICATE`.

In [ ]:
def deskew(crop_bgr):
    """
    Corregeix la inclinació d'un crop de matrícula.
    Retorna (aligned, angle_corr).
    """
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.sum(binary == 255) < np.sum(binary == 0):
        binary = cv2.bitwise_not(binary)
    points = np.column_stack(np.where(binary == 255))
    if len(points) < 5:
        return crop_bgr, 0.0
    rect = cv2.minAreaRect(points[:, ::-1].astype(np.float32))
    (cx, cy), (w, h), angle = rect
    angle_corr = angle + 90.0 if w < h else angle
    if abs(angle_corr) > ALIGN_ANGLE_MAX:
        return crop_bgr, 0.0
    H_img, W_img = crop_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((W_img / 2.0, H_img / 2.0), angle_corr, 1.0)
    aligned = cv2.warpAffine(crop_bgr, M, (W_img, H_img),
                             flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return aligned, angle_corr


crop_files = _glob_crops(PROCESSED_DIR)
if not crop_files:
    raise RuntimeError(
        f"No s'han trobat fitxers a '{PROCESSED_DIR}'. "
        "Cal executar primer el detector (Fase 01) per generar els crops."
    )

example_path = crop_files[EXAMPLE_INDEX % len(crop_files)]
crop_bgr     = cv2.imread(str(example_path))
aligned, angle_corr = deskew(crop_bgr)
angle_label = f'{angle_corr:+.1f}°' if angle_corr != 0.0 else '0.0° (no calia rotar)'
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original (pre-deskew)', fontsize=11)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Alineat — angle: {angle_label}', fontsize=11)
axes[1].axis('off')
plt.suptitle(f'Pas 0: Deskew — {example_path.name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Pas 1: Adaptive Threshold + MORPH_CLOSE

Binaritzem amb **Adaptive Threshold** (MEAN_C, BINARY_INV, `blockSize=31`, `C=15`) — els mateixos paràmetres del notebook de referència per mantenir compatibilitat.

$$T(x,y) = \text{mean}\big(\text{entorn}(x,y)\big) - C$$

**Millora: `cv2.MORPH_CLOSE` amb kernel vertical `(1, 3)`.**

El CLOSE és una dilatació seguida d'una erosió: **omple petits forats** i **connecta fragments** dins d'un mateix traç sense unir caràcters veïns. El kernel `(1×3)` actua exclusivament en la direcció vertical, cosa que:
- Consolida traços verticals fins que la binarització podria haver fragmentat (important per a la `'I'`).
- No actua horitzontalment, de manera que no fusiona caràcters adjacents.

Un kernel quadrat gran (p.ex. `3×3`) o horitzontal podria connectar lletres properes i crear bboxes fusionats.

In [ ]:
gray = cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)

thresh_raw = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    31, 15
)

# MORPH_CLOSE vertical (1×3): consolida traços fins sense fusionar caràcters veïns
morph_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, MORPH_KERNEL_SIZE)
thresh = cv2.morphologyEx(thresh_raw, cv2.MORPH_CLOSE, morph_kernel)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
axes[0].set_title('Entrada: alineada', fontsize=11)
axes[0].axis('off')
axes[1].imshow(thresh_raw, cmap='gray')
axes[1].set_title('Adaptive Threshold (raw)', fontsize=11)
axes[1].axis('off')
axes[2].imshow(thresh, cmap='gray')
axes[2].set_title(f'+ MORPH_CLOSE {MORPH_KERNEL_SIZE}', fontsize=11)
axes[2].axis('off')
plt.suptitle(f'Pas 1: Threshold + CLOSE — {example_path.name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Pas 2: Contorns externs (`RETR_EXTERNAL`)

**Canvi clau respecte a la versió anterior:** usem `cv2.RETR_EXTERNAL` en lloc de `cv2.RETR_TREE`.

| Mode | Comportament |
|------|-------------|
| `RETR_TREE` | Retorna **tots** els contorns: exterior + forats interiors (fills) |
| `RETR_EXTERNAL` | Retorna **únicament** els contorns més externs; ignora els forats |

Caràcters com `'O'`, `'0'`, `'D'`, `'A'`, `'B'` tenen forats tancats. Amb `RETR_TREE`, el forat interior genera un segon contorn amb un bounding box molt semblant al del caràcter complet: tots dos passen el filtre d'alçada mediana i el caràcter es compta **dues vegades**. `RETR_EXTERNAL` elimina directament els contorns interiors.

Continuem usant `CHAIN_APPROX_SIMPLE` per emmagatzemar menys punts (només els extrems de cada segment recte).

In [ ]:
# RETR_EXTERNAL: ignora els forats interiors (O, 0, D, A, B, etc.)
cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

overlay_all = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
for c in cnts:
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(overlay_all, (x, y), (x + w, y + h), (0, 0, 255), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(thresh, cmap='gray')
axes[0].set_title('Binari net (post-CLOSE)', fontsize=11)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(overlay_all, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Contorns RETR_EXTERNAL ({len(cnts)}) — vermell', fontsize=11)
axes[1].axis('off')
plt.suptitle(f'Pas 2: Contorns externs — {example_path.name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

tots_els_bboxes = [cv2.boundingRect(c) for c in cnts]
print(f"Contorns RETR_EXTERNAL detectats: {len(tots_els_bboxes)} (sense forats interiors)")

## Pas 3: Filtre per alçada mediana, aspect ratio i deduplicació IoU

**Filtre geomètric** (igual que al notebook original, amb un canvi):

1. Descartem brutícia minúscula (`h ≤ 10 px`).
2. Calculem la **mediana d'alçada** (estadístic robust).
3. Filtre d'alçada: $0.85 \cdot h_{\text{med}} < h < 1.15 \cdot h_{\text{med}}$
4. Filtre d'aspect ratio: $\mathbf{0.05} < w/h < 0.95$
   - **Límit inferior baixat a 0.05** (era 0.15) per no perdre la `'I'`, que pot tenir AR de 0.05–0.12.

**Deduplicació IoU** (xarxa de seguretat):

Encara que `RETR_EXTERNAL` elimina la majoria de forats, en casos extrems dos bboxes molt solapants poden sobreviure el filtre. L'IoU (Intersection over Union) detecta i elimina duplicats conservant el de **més àrea** (el contorn exterior):

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Si $\text{IoU} > \text{IOU\_OVERLAP\_THRESH} = 0.5$, eliminem el bbox de menys àrea.

In [ ]:
chars_filtrats = []

# Pas 3a: mediana d'alçada
altures_valides = [h for (x, y, w, h) in tots_els_bboxes if h > 10]
print(f"Alçades vàlides (h > 10 px): {len(altures_valides)}")

if altures_valides:
    h_mediana = np.median(altures_valides)
    print(f"Alçada mediana: {h_mediana:.1f} px")

    # Pas 3b: filtre alçada + AR (mínim AR = 0.05 per capturar la 'I')
    for (x, y, w, h) in tots_els_bboxes:
        ar = w / float(h)
        if (h_mediana * 0.85 < h < h_mediana * 1.15) and (0.05 < ar < 0.95):
            chars_filtrats.append((x, y, w, h))
    chars_filtrats.sort(key=lambda b: b[0])
else:
    print("No s'han trobat alçades vàlides.")

# Pas 3c: deduplicació per IoU
# Ordena per àrea descendent: el contorn exterior (més gran) entra primer i es conserva.
def _iou(a, b):
    ax, ay, aw, ah = a;  bx, by, bw, bh = b
    ix = max(0, min(ax + aw, bx + bw) - max(ax, bx))
    iy = max(0, min(ay + ah, by + bh) - max(ay, by))
    inter = ix * iy
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0

n_abans_iou = len(chars_filtrats)
kept = []
for bbox in sorted(chars_filtrats, key=lambda b: b[2] * b[3], reverse=True):
    if not any(_iou(bbox, k) > IOU_OVERLAP_THRESH for k in kept):
        kept.append(bbox)
chars_filtrats = sorted(kept, key=lambda b: b[0])
print(f"Bboxes pre-IoU: {n_abans_iou}  →  post-IoU: {len(chars_filtrats)}")

# Visualització
overlay_all  = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
overlay_filt = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in tots_els_bboxes:
    cv2.rectangle(overlay_all,  (x, y), (x + w, y + h), (0, 0, 255), 1)
for (x, y, w, h) in chars_filtrats:
    cv2.rectangle(overlay_filt, (x, y), (x + w, y + h), (0, 220, 0), 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(overlay_all,  cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Tots els contorns ({len(tots_els_bboxes)}) — vermell', fontsize=11)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(overlay_filt, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Filtrats + IoU ({len(chars_filtrats)}) — verd', fontsize=11)
axes[1].axis('off')
plt.suptitle(f'Pas 3: Filtre + IoU — {example_path.name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Caràcters retinguts: {len(chars_filtrats)}")

## Pas 4: Validació geomètrica (`is_plausible_plate`)

El filtre geomètric del pas anterior elimina soroll morfològic, però una imatge de soroll pot generar igualment un nombre de bboxes dins del rang [5, 8]. Afegim **comprovacions de coherència** que una matrícula real compleix però el soroll rarament:

| Comprovació | Llindar | Motivació |
|-------------|---------|----------|
| Recompte | $[N_{\min}, N_{\max}]$ | Matrícula vàlida |
| Uniformitat d'alçades | $\sigma_h / h_{\text{med}} \leq 20\%$ | Tots els caràcters d'una matrícula fan la mateixa alçada |
| Alineació vertical | $\sigma_{c_y} / h_{\text{med}} \leq 25\%$ | Els caràcters comparteixen línia base |
| Cobertura horitzontal | $(x_{\max} - x_{\min}) / W_{\text{img}} \geq 30\%$ | La matrícula ocupa la major part del crop |

Aquests llindars són **conservadors**: és preferible rebutjar algun crop dubtós que contaminar el dataset d'entrenament amb caràcters espuris.

In [ ]:
n_chars  = len(chars_filtrats)
H_img, W_img = aligned.shape[:2]
accepted = False
rejection_reason = ''

# Comprovació 1: rang de caràcters
if not (N_CHARS_MIN <= n_chars <= N_CHARS_MAX):
    rejection_reason = f'recompte={n_chars} fora de [{N_CHARS_MIN},{N_CHARS_MAX}]'

elif n_chars > 0:
    heights = [h for (x, y, w, h) in chars_filtrats]
    h_med   = float(np.median(heights))
    h_std   = float(np.std(heights))
    cys     = [y + h / 2. for (x, y, w, h) in chars_filtrats]
    cy_std  = float(np.std(cys))
    x_min   = min(x       for (x, y, w, h) in chars_filtrats)
    x_max   = max(x + w   for (x, y, w, h) in chars_filtrats)
    x_cov   = (x_max - x_min) / W_img

    # Comprovació 2: uniformitat d'alçades
    if h_med > 0 and h_std > HEIGHT_STD_MAX * h_med:
        rejection_reason = (f'alçades irregulars '
                            f'(std/med={h_std/h_med:.0%} > {HEIGHT_STD_MAX:.0%})')

    # Comprovació 3: alineació vertical dels centroides
    elif h_med > 0 and cy_std > ALIGN_CY_STD_MAX * h_med:
        rejection_reason = (f'mala alineació vertical '
                            f'(cy_std/med={cy_std/h_med:.0%} > {ALIGN_CY_STD_MAX:.0%})')

    # Comprovació 4: cobertura horitzontal
    elif x_cov < MIN_WIDTH_COVERAGE:
        rejection_reason = (f'amplada insuficient '
                            f'({x_cov:.0%} < {MIN_WIDTH_COVERAGE:.0%})')

    else:
        accepted = True

if accepted:
    print(f"✓ Crop ACCEPTAT: {n_chars} caràcters passen totes les comprovacions.")
else:
    print(f"✗ Crop REBUTJAT: {rejection_reason}")

## Pas 5: Copy & Resize a 28×28

Retallem cada caràcter de la imatge **threshold** (no de la BGR) i forcem la mida `28×28` amb `INTER_AREA`. Treballem des de `thresh` (post-CLOSE) per tenir els traços consolidats.

In [ ]:
caracters_per_cnn = []

if accepted and chars_filtrats:
    for (x, y, w, h) in chars_filtrats:
        char_crop    = thresh[y:y + h, x:x + w]
        char_resized = cv2.resize(char_crop, (CNN_INPUT_W, CNN_INPUT_H),
                                  interpolation=cv2.INTER_AREA)
        caracters_per_cnn.append(char_resized)

n_chars_ok = len(caracters_per_cnn)
n_cols = max(4, n_chars_ok if n_chars_ok > 0 else 1)
fig = plt.figure(figsize=(max(14, n_cols * 2.2), 6))

suffix      = f'  ✓ {n_chars_ok} caràcters' if accepted else f'  ✗ REBUTJAT ({rejection_reason})'
title_color = 'black' if accepted else 'orangered'
bbox_kw     = dict(facecolor='orange', alpha=0.25, pad=4) if not accepted else {}
fig.suptitle(f'{example_path.name}{suffix}', fontsize=12, fontweight='bold',
             color=title_color, bbox=bbox_kw)

ax1 = fig.add_subplot(2, n_cols, 1)
ax1.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
ax1.set_title('Original', fontsize=9); ax1.axis('off')

ax2 = fig.add_subplot(2, n_cols, 2)
ax2.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
ax2.set_title(f'Deskew ({angle_corr:+.1f}°)' if angle_corr != 0.0 else 'Deskew (0°)', fontsize=9)
ax2.axis('off')

ov_all  = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
ov_filt = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in tots_els_bboxes:
    cv2.rectangle(ov_all,  (x, y), (x + w, y + h), (0,   0, 255), 1)
for (x, y, w, h) in chars_filtrats:
    cv2.rectangle(ov_filt, (x, y), (x + w, y + h), (0, 220,   0), 2)

ax3 = fig.add_subplot(2, n_cols, 3)
ax3.imshow(cv2.cvtColor(ov_all, cv2.COLOR_BGR2RGB))
ax3.set_title(f'RETR_EXT ({len(tots_els_bboxes)})', fontsize=9); ax3.axis('off')

ax4 = fig.add_subplot(2, n_cols, 4)
ax4.imshow(cv2.cvtColor(ov_filt, cv2.COLOR_BGR2RGB))
ax4.set_title(f'Filtrats ({len(chars_filtrats)}) {"✓" if accepted else "✗"}',
              fontsize=9, color='green' if accepted else 'orangered')
ax4.axis('off')

for i, char_img in enumerate(caracters_per_cnn):
    ax = fig.add_subplot(2, n_cols, n_cols + i + 1)
    ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'char {i}', fontsize=8); ax.axis('off')

plt.tight_layout()
plt.show()

## Pas 6: Funcions reutilitzables i pipeline complet

Empaquem tota la lògica anterior en funcions i l'apliquem a tots els crops de `data/processed/`.

In [ ]:
def binarize_adaptive(aligned_bgr):
    """
    Adaptive Threshold (MEAN_C, BINARY_INV, 31, 15) + MORPH_CLOSE vertical.
    El CLOSE consolida traços fins (p.ex. 'I') sense fusionar caràcters adjacents.
    """
    gray = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2GRAY)
    th = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        31, 15,
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, MORPH_KERNEL_SIZE)
    return cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)


def extract_contours(thresh):
    """
    Extreu bboxes de contorns externs (RETR_EXTERNAL).
    RETR_EXTERNAL ignora els forats interiors de O, 0, D, A, B, etc.,
    evitant que el forat es compti com un caràcter separat.
    """
    cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [cv2.boundingRect(c) for c in cnts]


def filter_chars(bboxes):
    """
    Filtre d'alçada mediana (±15%) i aspect ratio [0.05, 0.95].
    AR mínim 0.05 (no 0.15) per no perdre la 'I' (AR ≈ 0.05–0.12).
    Retorna la llista filtrada ordenada per x.
    """
    heights = [h for (x, y, w, h) in bboxes if h > 10]
    if not heights:
        return []
    h_med = np.median(heights)
    result = [
        (x, y, w, h) for (x, y, w, h) in bboxes
        if (h_med * 0.85 < h < h_med * 1.15) and (0.05 < w / float(h) < 0.95)
    ]
    result.sort(key=lambda b: b[0])
    return result


def remove_overlapping_bboxes(bboxes, iou_thresh=IOU_OVERLAP_THRESH):
    """
    Elimina bboxes que solapen massa (IoU > iou_thresh).
    Conserva el de més àrea (el contorn exterior) i descarta el forat interior.
    Ordena per àrea descendent: el primer que entra a 'kept' sempre guanya.
    """
    if len(bboxes) <= 1:
        return bboxes

    def iou(a, b):
        ax, ay, aw, ah = a;  bx, by, bw, bh = b
        ix = max(0, min(ax + aw, bx + bw) - max(ax, bx))
        iy = max(0, min(ay + ah, by + bh) - max(ay, by))
        inter = ix * iy
        union = aw * ah + bw * bh - inter
        return inter / union if union > 0 else 0.0

    kept = []
    for bbox in sorted(bboxes, key=lambda b: b[2] * b[3], reverse=True):
        if not any(iou(bbox, k) > iou_thresh for k in kept):
            kept.append(bbox)
    kept.sort(key=lambda b: b[0])
    return kept


def is_plausible_plate(bboxes, img_w, img_h):
    """
    Comprova si els bboxes detectats són geomètricament coherents amb una matrícula.
    Retorna (True, '') si és vàlid, o (False, motiu) si és rebutjat.

    Comprovacions (conservadores: millor rebutjar que contaminar el dataset):
      1. Recompte dins de [N_CHARS_MIN, N_CHARS_MAX]
      2. Uniformitat d'alçades: std_h / h_med <= HEIGHT_STD_MAX
      3. Alineació vertical: std_cy / h_med <= ALIGN_CY_STD_MAX
      4. Cobertura horitzontal: (x_max - x_min) / img_w >= MIN_WIDTH_COVERAGE
    """
    n = len(bboxes)
    if not (N_CHARS_MIN <= n <= N_CHARS_MAX):
        return False, f'recompte={n} fora de [{N_CHARS_MIN},{N_CHARS_MAX}]'

    heights = [h for (x, y, w, h) in bboxes]
    h_med   = float(np.median(heights))
    h_std   = float(np.std(heights))
    cys     = [y + h / 2. for (x, y, w, h) in bboxes]
    cy_std  = float(np.std(cys))
    x_min   = min(x       for (x, y, w, h) in bboxes)
    x_max   = max(x + w   for (x, y, w, h) in bboxes)
    x_cov   = (x_max - x_min) / img_w if img_w > 0 else 0.0

    if h_med > 0 and h_std > HEIGHT_STD_MAX * h_med:
        return False, (f'alçades irregulars '
                       f'(std/med={h_std/h_med:.0%} > {HEIGHT_STD_MAX:.0%})')
    if h_med > 0 and cy_std > ALIGN_CY_STD_MAX * h_med:
        return False, (f'mala alineació vertical '
                       f'(cy_std/med={cy_std/h_med:.0%} > {ALIGN_CY_STD_MAX:.0%})')
    if x_cov < MIN_WIDTH_COVERAGE:
        return False, (f'amplada insuficient '
                       f'({x_cov:.0%} < {MIN_WIDTH_COVERAGE:.0%})')
    return True, ''


def crops_and_resize(thresh, bboxes):
    """Retalla i redimensiona cada bbox a CNN_INPUT_W × CNN_INPUT_H (28×28)."""
    return [
        cv2.resize(thresh[y:y + h, x:x + w], (CNN_INPUT_W, CNN_INPUT_H),
                   interpolation=cv2.INTER_AREA)
        for (x, y, w, h) in bboxes
    ]


def parse_crop_filename(path):
    """Extreu (stem_base, idx) de noms '{stem}_cand{n}.jpg' o '{stem}_box{n}.png'."""
    for pattern in [r'^(.+)_cand(\d+)$', r'^(.+)_box(\d+)$']:
        m = re.match(pattern, path.stem)
        if m:
            return m.group(1), int(m.group(2))
    return None, None


def visualize_plate(crop_bgr, aligned, angle_corr,
                    thresh, all_bboxes, filtered_bboxes,
                    chars_28, title='', accepted=True, rejection_reason=''):
    """
    Figura diagnòstica completa.
      Fila 1: original | deskew | RETR_EXTERNAL | filtrats+IoU
      Fila 2: caràcters 28×28
    Si rejected, mostra el motiu al títol.
    """
    n      = len(chars_28)
    n_cols = max(4, n if n > 0 else 1)
    fig    = plt.figure(figsize=(max(14, n_cols * 2.2), 6))

    if accepted:
        suffix = f'  ✓ {n} caràcters'
    else:
        suffix = f'  ✗ REBUTJAT ({rejection_reason})'
    title_clr = 'black' if accepted else 'orangered'
    bbox_kw   = dict(facecolor='orange', alpha=0.25, pad=4) if not accepted else {}
    fig.suptitle(title + suffix, fontsize=11, fontweight='bold',
                 color=title_clr, bbox=bbox_kw)

    ax1 = fig.add_subplot(2, n_cols, 1)
    ax1.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    ax1.set_title('Original', fontsize=9); ax1.axis('off')

    ax2 = fig.add_subplot(2, n_cols, 2)
    ax2.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
    ax2.set_title(f'Deskew ({angle_corr:+.1f}°)' if angle_corr != 0.0
                  else 'Deskew (0° — ok)', fontsize=9)
    ax2.axis('off')

    ov_all  = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
    ov_filt = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
    for (x, y, w, h) in all_bboxes:
        cv2.rectangle(ov_all,  (x, y), (x + w, y + h), (0,   0, 255), 1)
    for (x, y, w, h) in filtered_bboxes:
        cv2.rectangle(ov_filt, (x, y), (x + w, y + h), (0, 220,   0), 2)

    ax3 = fig.add_subplot(2, n_cols, 3)
    ax3.imshow(cv2.cvtColor(ov_all, cv2.COLOR_BGR2RGB))
    ax3.set_title(f'RETR_EXT ({len(all_bboxes)})', fontsize=9); ax3.axis('off')

    ax4 = fig.add_subplot(2, n_cols, 4)
    ax4.imshow(cv2.cvtColor(ov_filt, cv2.COLOR_BGR2RGB))
    ax4.set_title(f'Filtrats ({len(filtered_bboxes)}) {"✓" if accepted else "✗"}',
                  fontsize=9, color='green' if accepted else 'orangered')
    ax4.axis('off')

    for i, char_img in enumerate(chars_28):
        ax = fig.add_subplot(2, n_cols, n_cols + i + 1)
        ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'char {i}', fontsize=8); ax.axis('off')

    plt.tight_layout()
    plt.show()


print("Funcions definides: deskew, binarize_adaptive, extract_contours, filter_chars,")
print("  remove_overlapping_bboxes, is_plausible_plate, crops_and_resize,")
print("  parse_crop_filename, visualize_plate")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
crop_files = _glob_crops(PROCESSED_DIR)

if not crop_files:
    print(f"No s'han trobat crops a '{PROCESSED_DIR}'.")
else:
    print(f"Crops trobats    : {len(crop_files)}")
    print(f"Directori sortida: {OUTPUT_DIR.resolve()}")
    print("-" * 68)

    stats = {'accepted': 0, 'rejected': 0, 'skipped': 0}

    for crop_path in crop_files:
        stem, cand_idx = parse_crop_filename(crop_path)
        if stem is None:
            print(f"  ? {crop_path.name:40s} → nom inesperat (SALTAT)")
            stats['skipped'] += 1
            continue

        crop_bgr_i = cv2.imread(str(crop_path))
        if crop_bgr_i is None:
            print(f"  ! {crop_path.name:40s} → error de lectura (SALTAT)")
            stats['skipped'] += 1
            continue

        H_i, W_i = crop_bgr_i.shape[:2]

        # ── Pipeline complet ──────────────────────────────────────────────────
        aligned_i, angle_i = deskew(crop_bgr_i)
        thresh_i            = binarize_adaptive(aligned_i)
        all_bboxes_i        = extract_contours(thresh_i)
        filtered_i          = filter_chars(all_bboxes_i)
        filtered_i          = remove_overlapping_bboxes(filtered_i)
        accepted_i, reason_i = is_plausible_plate(filtered_i, W_i, H_i)
        chars_28_i          = crops_and_resize(thresh_i, filtered_i) if accepted_i else []

        if accepted_i:
            for idx, char_img in enumerate(chars_28_i):
                out_name = f'{crop_path.stem}_char{idx}.png'
                cv2.imwrite(str(OUTPUT_DIR / out_name), char_img)
            angle_str = f'{angle_i:+.1f}°'
            print(f"  ✓ {crop_path.name:40s} angle={angle_str:>6}  → {len(filtered_i)} chars")
            stats['accepted'] += 1
        else:
            angle_str = f'{angle_i:+.1f}°'
            print(f"  ✗ {crop_path.name:40s} angle={angle_str:>6}  → {reason_i}")
            stats['rejected'] += 1

    print("-" * 68)
    total = sum(stats.values())
    print(f"\nResum: acceptats={stats['accepted']}  rebutjats={stats['rejected']}  "
          f"saltats={stats['skipped']}  total={total}")
    print(f"Caràcters desats a: {OUTPUT_DIR.resolve()}")

## Visualització d'exemples acceptats

Mostrem fins a 4 matrícules acceptades amb els seus caràcters segmentats.

In [ ]:
char0_files = sorted(OUTPUT_DIR.glob('*_char0.png'))[:4]

if not char0_files:
    print(f"No s'han trobat caràcters a '{OUTPUT_DIR}'. Executa primer el bucle principal.")
else:
    for char0_path in char0_files:
        prefix = re.sub(r'_char0$', '', char0_path.stem)

        crop_path_i = None
        for ext in ('.jpg', '.png'):
            candidate = PROCESSED_DIR / f'{prefix}{ext}'
            if candidate.exists():
                crop_path_i = candidate
                break

        if crop_path_i is None:
            print(f"Crop original no trobat: {prefix}")
            continue

        crop_bgr_i           = cv2.imread(str(crop_path_i))
        H_i, W_i             = crop_bgr_i.shape[:2]
        aligned_i, angle_i   = deskew(crop_bgr_i)
        thresh_i             = binarize_adaptive(aligned_i)
        all_bb_i             = extract_contours(thresh_i)
        filt_i               = filter_chars(all_bb_i)
        filt_i               = remove_overlapping_bboxes(filt_i)
        ok_i, reason_i       = is_plausible_plate(filt_i, W_i, H_i)
        chars_i              = crops_and_resize(thresh_i, filt_i) if ok_i else []

        visualize_plate(
            crop_bgr_i, aligned_i, angle_i,
            thresh_i, all_bb_i, filt_i, chars_i,
            title=prefix, accepted=ok_i, rejection_reason=reason_i,
        )

## Debug: totes les fases per a 20 crops

Per a cada crop es mostren **6 panells** a la fila superior i els **caràcters 28×28** a la fila inferior:

| Panell | Contingut |
|--------|-----------|
| 1 | Crop original BGR |
| 2 | Post-deskew BGR |
| 3 | `thresh` post-MORPH_CLOSE |
| 4 | Contorns `RETR_EXTERNAL` (vermell) |
| 5 | Bboxes filtrats + deduplicats (verd=acceptat / taronja=rebutjat) |
| 6 | Info: angle, recomptes, motiu de rebuig |

Els crops es trien aleatòriament (llavor 42) per garantir diversitat.

In [ ]:
all_crops = _glob_crops(PROCESSED_DIR)

if not all_crops:
    print(f"No s'han trobat crops a '{PROCESSED_DIR}'.")
else:
    rng = np.random.default_rng(42)
    if len(all_crops) > N_DEBUG:
        idxs = sorted(rng.choice(len(all_crops), N_DEBUG, replace=False))
        debug_files = [all_crops[i] for i in idxs]
    else:
        debug_files = all_crops

    print(f"Crops totals: {len(all_crops)}  |  Mostrant: {len(debug_files)}")
    print("=" * 72)

    for i_dbg, crop_path in enumerate(debug_files):
        crop_bgr_d = cv2.imread(str(crop_path))
        if crop_bgr_d is None:
            print(f"  [{i_dbg+1:02d}] ERROR llegint {crop_path.name}")
            continue

        H_d, W_d = crop_bgr_d.shape[:2]

        # ── Pipeline pas a pas ────────────────────────────────────────────────
        aligned_d, angle_d = deskew(crop_bgr_d)
        thresh_d            = binarize_adaptive(aligned_d)
        all_bb_d            = extract_contours(thresh_d)
        filt_d              = filter_chars(all_bb_d)
        filt_d_dedup        = remove_overlapping_bboxes(filt_d)
        accepted_d, reason_d = is_plausible_plate(filt_d_dedup, W_d, H_d)
        chars_d             = crops_and_resize(thresh_d, filt_d_dedup) if accepted_d else []

        n2_d   = len(filt_d_dedup)
        n_cols = max(6, max(len(chars_d), 1))
        fig    = plt.figure(figsize=(max(15, n_cols * 2.1), 5.5))

        color_t = 'darkgreen' if accepted_d else 'orangered'
        status  = (f'✓ {n2_d} chars' if accepted_d
                   else f'✗ {reason_d}')
        fig.suptitle(
            f'[{i_dbg+1:02d}/{len(debug_files)}]  {crop_path.name}'
            f'  |  angle={angle_d:+.1f}°  |  {status}',
            fontsize=10, fontweight='bold', color=color_t
        )

        # ── Panell 1: Original ────────────────────────────────────────────────
        ax = fig.add_subplot(2, n_cols, 1)
        ax.imshow(cv2.cvtColor(crop_bgr_d, cv2.COLOR_BGR2RGB))
        ax.set_title('Original', fontsize=8, pad=3); ax.axis('off')

        # ── Panell 2: Post-deskew ─────────────────────────────────────────────
        ax = fig.add_subplot(2, n_cols, 2)
        ax.imshow(cv2.cvtColor(aligned_d, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Deskew ({angle_d:+.1f}°)', fontsize=8, pad=3); ax.axis('off')

        # ── Panell 3: thresh post-CLOSE ───────────────────────────────────────
        ax = fig.add_subplot(2, n_cols, 3)
        ax.imshow(thresh_d, cmap='gray')
        ax.set_title(f'thresh+CLOSE', fontsize=8, pad=3); ax.axis('off')

        # ── Panell 4: RETR_EXTERNAL ───────────────────────────────────────────
        ax  = fig.add_subplot(2, n_cols, 4)
        vis4 = cv2.cvtColor(thresh_d, cv2.COLOR_GRAY2BGR)
        for (x, y, w, h) in all_bb_d:
            cv2.rectangle(vis4, (x, y), (x + w, y + h), (0, 0, 255), 1)
        ax.imshow(cv2.cvtColor(vis4, cv2.COLOR_BGR2RGB))
        ax.set_title(f'RETR_EXT ({len(all_bb_d)})', fontsize=8, pad=3); ax.axis('off')

        # ── Panell 5: Filtrats + deduplicats ──────────────────────────────────
        ax  = fig.add_subplot(2, n_cols, 5)
        vis5 = cv2.cvtColor(thresh_d, cv2.COLOR_GRAY2BGR)
        col5 = (0, 210, 0) if accepted_d else (0, 100, 255)
        for (x, y, w, h) in filt_d_dedup:
            cv2.rectangle(vis5, (x, y), (x + w, y + h), col5, 2)
        ax.imshow(cv2.cvtColor(vis5, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Filtrats ({n2_d})', fontsize=8, pad=3,
                     color='darkgreen' if accepted_d else 'orangered')
        ax.axis('off')

        # ── Panell 6: Info ────────────────────────────────────────────────────
        ax = fig.add_subplot(2, n_cols, 6)
        info = [
            f'angle  : {angle_d:+.2f}°',
            f'REXT   : {len(all_bb_d)} bboxes',
            f'filtre : {len(filt_d)} bboxes',
            f'IoU    : {n2_d} bboxes',
            f'mida   : {W_d}×{H_d} px',
            '',
            '✓ ACCEPTAT' if accepted_d else f'✗ REBUTJAT',
        ]
        if not accepted_d:
            info.append(reason_d)
        ax.text(0.05, 0.5, '\n'.join(info), transform=ax.transAxes,
                fontsize=7.5, va='center', fontfamily='monospace',
                color='darkgreen' if accepted_d else 'darkred')
        ax.set_title('Info', fontsize=8, pad=3)
        ax.set_facecolor('#f5f5f5'); ax.axis('off')

        # ── Fila 2: caràcters 28×28 ───────────────────────────────────────────
        for i_c, char_img in enumerate(chars_d):
            ax = fig.add_subplot(2, n_cols, n_cols + i_c + 1)
            ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
            ax.set_title(f'c{i_c}', fontsize=7, pad=2); ax.axis('off')

        if not chars_d:
            ax = fig.add_subplot(2, n_cols, n_cols + 1)
            ax.text(0.5, 0.5, 'cap caràcter\nacceptat',
                    ha='center', va='center', fontsize=9, color='gray')
            ax.axis('off')

        plt.tight_layout(rect=[0, 0, 1, 0.91])
        plt.show()

        print(f"  [{i_dbg+1:02d}] {crop_path.name:38s}"
              f"  angle={angle_d:+.1f}°  REXT={len(all_bb_d):3d}"
              f"  filt={n2_d:2d}  {'✓ OK' if accepted_d else f'✗ {reason_d}'}")

    print("=" * 72)
    print("Debug completat.")

## TODO: Post-filtrat per confiança de l'OCR

L'OCR sobre els caràcters 28×28 encara no està implementat. Quan ho estigui, s'haurà d'afegir un filtre de confiança post-OCR com a darrera etapa del pipeline:

### Estratègia prevista

**1. Filtre per score de confiança per caràcter**

La majoria de models d'OCR (EMNIST, Tesseract, models propis) produeixen un score de confiança per a cada predicció. Es descartaran els caràcters amb score inferior a un llindar (p.ex. 0.6 o 0.7), que s'haurà de calibrar empíricament sobre un conjunt de validació.

**2. No es valida per patró de format**

El dataset conté matrícules espanyoles de formats diversos (antigues provincials, modernes, de vehicles especials). Com que **no hi ha un patró fix** (p.ex. NNNN-LLL), **no es pot implementar un filtre de format** sense conèixer prèviament el tipus de matrícula. El filtre haurà de basar-se exclusivament en la confiança per caràcter, no en la seqüència de números i lletres.

**3. Descart de la matrícula sencera si resten pocs caràcters**

Si després del filtrat per confiança queden menys de `N_CHARS_MIN = 5` caràcters, el crop sencer es descartarà: una matrícula amb massa caràcters il·legibles no és útil per a l'entrenament ni per al reconeixement.

### Lloc al pipeline

```
... → is_plausible_plate → resize 28×28
  → [OCR] model.predict → scores per caràcter
  → filtre score < llindar  ← (per implementar)
  → si queden < N_CHARS_MIN → descartar  ← (per implementar)
  → guardar caràcters acceptats
```